
## Q1. What is Unity Catalog?


- **Unity Catalog** is the `centralized governance solution` in Databricks. 
- It manages `data access, permissions, security, auditing` and `data lineage`.


**For example:**
   - In my E-Commerce Lakehouse project, 
       - I can give data engineers access to Bronze and Silver tables while allowing business analysts to read only Gold tables.


**SQL example:**


```sql
SHOW CATALOGS;
```


This displays the catalogs available in the Unity Catalog metastore.


## Q2. What is the Unity Catalog hierarchy?


**Unity Catalog follows a hierarchy:**

- **Metastore** — Top-level governance container.
- **Catalog** — Collection of schemas.
- **Schema** — Collection of tables, views and volumes.
- **Table / View / Volume** — Data objects.



**SQL example:**


```sql
-- Create a catalog
CREATE CATALOG IF NOT EXISTS ecommerce;

-- Create schemas
CREATE SCHEMA IF NOT EXISTS ecommerce.bronze;

CREATE SCHEMA IF NOT EXISTS ecommerce.silver;

CREATE SCHEMA IF NOT EXISTS ecommerce.gold;

-- Display schemas
SHOW SCHEMAS IN ecommerce;

-- Display tables
SHOW TABLES IN ecommerce.bronze;
```


**My project example:**


```text
workspace.ecommerce_bronze.customers
workspace.ecommerce_silver.customers
workspace.ecommerce_gold.daily_sales
```


**The naming format is:**


```text
catalog.schema.table
```


## Q3. What is the difference between a Catalog and a Schema?


A **catalog** is a `higher-level container` that **organizes schemas**, whereas a schema organizes t`ables, views, volumes` and `other database objects`.

**For example:**
 -     `workspace` is my catalog, and `ecommerce_bronze`,
 - `ecommerce_silver` and `ecommerce_gold` are schemas inside it.


**SQL example:**


```sql
CREATE SCHEMA IF NOT EXISTS workspace.ecommerce_bronze;

CREATE TABLE IF NOT EXISTS workspace.ecommerce_bronze.customers (
    CustomerID INT,
    CustomerName STRING,
    Email STRING
);

SHOW TABLES IN workspace.ecommerce_bronze;
```



## Q4. How do you manage permissions in Unity Catalog?


I use **Unity Catalog privileges** such as `USE CATALOG`, `USE SCHEMA`, `SELECT` and `MODIFY` to control data access.

**For example:**
   - I can grant analysts read-only access to Gold tables while       restricting access to sensitive customer and payment information.


**SQL example — Grant read access:**


```sql
GRANT USE CATALOG
ON CATALOG workspace
TO `data_analysts`;

GRANT USE SCHEMA
ON SCHEMA workspace.ecommerce_gold
TO `data_analysts`;

GRANT SELECT
ON SCHEMA workspace.ecommerce_gold
TO `data_analysts`;
```


**Revoke access:**


```sql
REVOKE SELECT
ON SCHEMA workspace.ecommerce_gold
FROM `data_analysts`;
```


**Check permissions:**


```sql
SHOW GRANTS ON SCHEMA workspace.ecommerce_gold;
```


## Q5. What is Row-Level Security?


- **Row-Level Security (RLS)** is a `data security mechanism` that `restricts which rows a user can access` based on their i**dentity, role, department, region** or **permissions**.
- In Databricks, Unity Catalog supports **RLS** through `row filters`, which apply `filtering rules` whenever users query a table.


**SQL example:**


**Step 1 — Create a security function:**


```sql
CREATE OR REPLACE FUNCTION
workspace.ecommerce_gold.region_filter(region STRING)

RETURNS BOOLEAN

RETURN
    CASE
        WHEN IS_ACCOUNT_GROUP_MEMBER('ecommerce_admins')
            THEN TRUE

        WHEN IS_ACCOUNT_GROUP_MEMBER('chennai_team')
             AND region = 'Chennai'
            THEN TRUE

        WHEN IS_ACCOUNT_GROUP_MEMBER('bangalore_team')
             AND region = 'Bangalore'
            THEN TRUE

        ELSE FALSE
    END;
```


**Step 2 — Apply the row filter:**


```sql
ALTER TABLE workspace.ecommerce_gold.daily_sales

SET ROW FILTER
workspace.ecommerce_gold.region_filter

ON (Region);
```


**Step 3 — Query the table:**


```sql
SELECT *
FROM workspace.ecommerce_gold.daily_sales;
```


The results are filtered according to the querying user's group membership.


## Q6. What is Column-Level Security?


- **Column-Level Security (CLS)** is a `data security mechanism` that controls which `column values a user can view` based on their **identity, role, group membership or permissions**.
- In Databricks, Unity Catalog supports **column-level security** through column masks. 
- A column mask uses a SQL **user-defined function (UDF)** to return either the original value or a masked value whenever the column is queried.



**SQL example:**


**Step 1 — Create a masking function:**




```sql
CREATE OR REPLACE FUNCTION
workspace.ecommerce_gold.mask_email(email STRING)

RETURNS STRING

RETURN
    CASE
        WHEN IS_ACCOUNT_GROUP_MEMBER('ecommerce_admins')
            THEN email

        ELSE CONCAT(
            'XXXXXX',
            SUBSTRING(email, INSTR(email, '@'))
        )
    END;
```


**Step 2 — Apply the column mask:**


```sql
ALTER TABLE workspace.ecommerce_gold.customers

ALTER COLUMN Email

SET MASK workspace.ecommerce_gold.mask_email;
```


**Step 3 — Query the table:**


```sql
SELECT
    CustomerID,
    CustomerName,
    Email
FROM workspace.ecommerce_gold.customers;
```


Administrators see the original email, while other users see the masked value.


## Q7. What is the difference between Row-Level Security and Column-Level Security?


- **Row-Level Security (RLS)** is a `data security mechanism` that controls which `rows a user can access` based on their i**dentity, role, department, region** or **permissions**.
- **Column-Level Security (CLS)** is a `data security mechanism` that controls which `column values a user can view` based on their **identity, role, group membership or permissions**.



**SQL examples:**


**Row-level security:**


```sql
ALTER TABLE workspace.ecommerce_gold.daily_sales

SET ROW FILTER
workspace.ecommerce_gold.region_filter

ON (Region);
```


**Column-level security:**


```sql
ALTER TABLE workspace.ecommerce_gold.customers

ALTER COLUMN Email

SET MASK workspace.ecommerce_gold.mask_email;
```


## Q8. What is Data Lineage?


**Data lineage** is used to `tracks where data comes from`, `how it is transformed` and `where it is consumed`.


**In my E-Commerce Lakehouse project:**

-  I can trace daily sales from the Gold table back to Silver orders and the original Bronze data.


**SQL example — Inspect the Gold table:**


```sql
DESCRIBE TABLE EXTENDED
workspace.ecommerce_gold.daily_sales;
```


**Query the Gold data:**


```sql
SELECT *
FROM workspace.ecommerce_gold.daily_sales;
```


**To inspect actual upstream and downstream lineage, I use Catalog Explorer:**


```text
Catalog
   |
workspace
   |
ecommerce_gold
   |
daily_sales
   |
Lineage tab
```


If a dashboard displays incorrect revenue, I trace the dependencies backward and investigate the source tables and transformations.


## Q9. What is an External Location?


- An **External Location** is a` Unity Catalog securable object` that combines a `cloud storage path` with a `storage credential`.
- It provides **access** to `external cloud storage` such as **AWS S3, Azure Data Lake Storage or Google Cloud Storage**.


**SQL example:**


First, an administrator must configure a storage credential.


**Then create the external location:**


```sql
CREATE EXTERNAL LOCATION ecommerce_landing_location

URL 'abfss://landing@storageaccount.dfs.core.windows.net/ecommerce'

WITH (
    STORAGE CREDENTIAL ecommerce_storage_credential
);
```


**Grant read access:**


```sql
GRANT READ FILES

ON EXTERNAL LOCATION ecommerce_landing_location

TO `data_engineers`;
```


**For write access:**


```sql
GRANT READ FILES, WRITE FILES

ON EXTERNAL LOCATION ecommerce_landing_location

TO `data_engineers`;
```


The storage credential must already exist and have the appropriate cloud permissions.



## Q10. What is a Volume in Unity Catalog?



A **Volume** is a `Unity Catalog object` used to **store and govern** access to `non-tabular files` such as **CSV, JSON, Parquet, images and documents**.


**In my E-Commerce Lakehouse project:**

-      I use a Volume to store landing files, 
-      schema information and checkpoint files. 
-      Delta tables store the processed Bronze, Silver and Gold data.


**SQL example — Create a managed Volume:**


```sql
CREATE VOLUME IF NOT EXISTS
workspace.default.data_volume;
```


**Grant access:**


```sql
GRANT READ VOLUME

ON VOLUME workspace.default.data_volume

TO `data_engineers`;
```


```sql
GRANT WRITE VOLUME

ON VOLUME workspace.default.data_volume

TO `data_engineers`;
```


**Read a CSV file using PySpark:**


```python
df = (
    spark.read
    .option("header", "true")
    .csv(
        "/Volumes/workspace/default/data_volume/"
        "ecommerce_lakehouse/landing/customers/customers.csv"
    )
)

display(df)
```


## Q11. What is the difference between a Volume and a Table?


| Feature | Table | Volume |
|---|---|---|
| Purpose | Stores structured data | Stores files |
| Data format | Rows and columns | Files and directories |
| Data access | SQL and DataFrame APIs | File-system APIs and file paths |
| Examples | Customers, orders, payments | CSV, JSON, PDF, images |
| Operations | SELECT, INSERT, UPDATE, DELETE, MERGE | Read, write, upload, download, delete files |
| Governance | Table-level permissions | Volume-level permissions |
| Storage | Managed or external table storage | Managed or external volume storage |
| Common use | Analytics and reporting | Raw files and unstructured data |


**SQL example — Create a table:**


```sql
CREATE TABLE IF NOT EXISTS
workspace.ecommerce_bronze.customers (
    CustomerID INT,
    CustomerName STRING,
    Email STRING
);
```


**Query the table:**


```sql
SELECT *
FROM workspace.ecommerce_bronze.customers;
```


**Volume path:**


```text
/Volumes/workspace/default/data_volume/
```


**Table name:**


```text
workspace.ecommerce_bronze.customers
```



## Q12. What is a Connection in Unity Catalog?


- A **Connection in Unity Catalog** is a `securable object` that stores the `configuration and authentication details` required for Databricks to connect to an `external data system`.
- It allows Databricks to `securely communicate with external databases and services` without **hardcoding connection credentials** into **notebooks or SQL queries**.


**SQL example — Create a MySQL connection:**


```sql
CREATE CONNECTION mysql_ecommerce_connection

TYPE MYSQL

OPTIONS (
    host 'mysql-server.example.com',
    port '3306',
    user secret('ecommerce_scope', 'mysql_user'),
    password secret('ecommerce_scope', 'mysql_password')
);
```


The secret scope and keys must already exist. I use secrets instead of hardcoding database credentials.


**Display connections:**


```sql
SHOW CONNECTIONS;
```



## Q13. What is Lakehouse Federation?


- **Lakehouse Federation** is a `Databricks feature` that allows users to `access and query` data from **external databases** and **catalogs** without `migrating` or `copying` the data into Databricks.
- It integrates with Unity Catalog to provide centralized governance, access control and auditing for external data.


**For example:**

-    - if customer information is stored in MySQL, 
-    - I can configure a connection and a foreign catalog, 
-    - then query the MySQL tables using Databricks SQL.


**SQL example:**


**First, create the connection:**


**Query the external table:**



```sql
CREATE CONNECTION mysql_ecommerce_connection

TYPE MYSQL

OPTIONS (
    host 'mysql-server.example.com',
    port '3306',
    user secret('ecommerce_scope', 'mysql_user'),
    password secret('ecommerce_scope', 'mysql_password')
);
```


**Then create a foreign catalog:**


```sql
CREATE FOREIGN CATALOG mysql_ecommerce

USING CONNECTION mysql_ecommerce_connection

OPTIONS (
    database 'ecommerce_db'
);
```


```sql
SELECT *
FROM mysql_ecommerce.public.customers;
```


The remote schema name depends on the MySQL database metadata exposed through the connector.


## Q14. What is a Foreign Catalog?


A **Foreign Catalog** is a` Unity Catalog object` that represents the `schemas and tables of an external database`.

**For example:**
 - I can register a MySQL database as a foreign catalog and query its customer and order tables using three-level naming.


**SQL example:**


```sql
CREATE FOREIGN CATALOG mysql_ecommerce

USING CONNECTION mysql_ecommerce_connection

OPTIONS (
    database 'ecommerce_db'
);
```


**Display schemas:**


```sql
SHOW SCHEMAS IN mysql_ecommerce;
```


**Display tables:**



```sql
SHOW TABLES IN mysql_ecommerce.public;
```


**Query remote data:**


```sql
SELECT *
FROM mysql_ecommerce.public.customers;
```


The actual table data remains in MySQL. Databricks accesses it through Lakehouse Federation.


## Q15. What is the difference between a Foreign Catalog and an External Location?


- A **Foreign Catalog** is used to access tables in external databases such as MySQL and PostgreSQL through federation.
- An **External Location** is used to govern access to external cloud storage paths such as S3 and Azure Data Lake Storage.


**SQL examples:**


**Foreign Catalog:**


```sql
SELECT *
FROM mysql_ecommerce.public.orders;
```


**External Location:**


```sql
CREATE EXTERNAL LOCATION ecommerce_landing_location

URL 'abfss://landing@storageaccount.dfs.core.windows.net/ecommerce'

WITH (
    STORAGE CREDENTIAL ecommerce_storage_credential
);
```



For example, I use a Foreign Catalog to query MySQL orders and an External Location to govern access to incoming CSV files stored in cloud storage.